# CIFAR-10 Classification Results Comparison
### Date: 2025-07-09
### Purpose: Compare training/test loss and accuracy across different sweep runs

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Define the sweep directories
results_dir = Path("results")
sweep_dirs = [
    "sweep_20250708_581384.opbs_0",
    "sweep_20250708_581385.opbs_1",
    "sweep_20250708_581386.opbs_2",
    "sweep_20250708_581387.opbs_3",
    "sweep_20250708_581388.opbs_4",
    "sweep_20250708_581389.opbs_5",
    "sweep_20250708_581390.opbs_6",
    "sweep_20250708_581391.opbs_7",
    "sweep_20250708_581392.opbs_8",
    "sweep_20250708_581393.opbs_9",
    "sweep_20250708_581394.opbs_10",
    "sweep_20250708_581395.opbs_11",
    "sweep_20250708_581396.opbs_12",
    "sweep_20250708_581397.opbs_13",
    "sweep_20250708_581398.opbs_14",
    "sweep_20250708_581399.opbs_15",
    "sweep_20250708_581400.opbs_16",
    "sweep_20250708_581401.opbs_17"
]

print(f"Found {len(sweep_dirs)} sweep directories")

In [ ]:
# Function to load results from a sweep directory
def load_sweep_results(sweep_dir):
    sweep_path = results_dir / sweep_dir
    
    # Check if my_akorn_cifar10_final.pth exists
    model_path = sweep_path / "my_akorn_cifar10_final.pth"
    if not model_path.exists():
        return None
    
    # Load parameters
    params_path = sweep_path / "parameters.json"
    if params_path.exists():
        with open(params_path, 'r') as f:
            params = json.load(f)
    else:
        params = {}
    
    # Try to load the model checkpoint to extract metrics
    try:
        checkpoint = torch.load(model_path, map_location='cpu')
        
        # Extract metrics from checkpoint if available
        result = {
            'sweep_dir': sweep_dir,
            'run_id': sweep_dir.split('_')[-1],
            'model_exists': True
        }
        
        # Add parameters
        result.update(params)
        
        # Try to extract metrics from checkpoint
        if 'train_loss' in checkpoint['history']:
            result['final_train_loss'] = checkpoint['history']['train_loss'][-1]
        if 'train_acc' in checkpoint['history']:
            result['final_train_acc'] = checkpoint['history']['train_acc'][-1]
        if 'test_loss' in checkpoint['history']:
            result['final_test_loss'] = checkpoint['history']['test_loss'][-1]
        if 'test_acc' in checkpoint['history']:
            result['final_test_acc'] = checkpoint['history']['test_acc'][-1]
        # if 'epoch' in checkpoint:
        #     result['final_epoch'] = checkpoint['epoch']
        
        return result
        
    except Exception as e:
        print(f"Error loading {model_path}: {e}")
        return {
            'sweep_dir': sweep_dir,
            'run_id': sweep_dir.split('_')[-1],
            'model_exists': True,
            'error': str(e)
        }

# Load all results
all_results = []
for sweep_dir in sweep_dirs:
    result = load_sweep_results(sweep_dir)
    if result is not None:
        all_results.append(result)
    else:
        # Add entry for missing model
        all_results.append({
            'sweep_dir': sweep_dir,
            'run_id': sweep_dir.split('_')[-1],
            'model_exists': False
        })

print(f"Loaded results from {len(all_results)} sweep runs")
print(f"Models found: {sum(1 for r in all_results if r.get('model_exists', False))}")

In [ ]:
# Create a comprehensive DataFrame
df = pd.DataFrame(all_results)

# Create parameter labels for better visualization
df['param_label'] = df.apply(lambda row: f"T={row['T']:.0f}, γ={row['gamma']:.2f}" if pd.notna(row.get('T')) and pd.notna(row.get('gamma')) else "N/A", axis=1)

# Display basic information
print("Dataset Summary:")
print(f"Total runs: {len(df)}")
print(f"Runs with models: {df['model_exists'].sum()}")
print(f"Runs without models: {(~df['model_exists']).sum()}")

# Display the first few rows
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Create summary table of key metrics
summary_columns = ['param_label', 'T', 'gamma', 'model_exists', 'final_train_loss', 'final_train_acc', 
                  'final_test_loss', 'final_test_acc']

# Filter for available columns
available_columns = [col for col in summary_columns if col in df.columns]
summary_df = df[available_columns].copy()

# Sort by T then gamma for better readability
summary_df = summary_df.sort_values(['T', 'gamma'])

print("Summary of Results:")
print("=" * 50)
summary_df

In [ ]:
df.columns

In [ ]:
# Create visualizations if metrics are available
metrics_available = any(col in df.columns for col in ['final_train_loss', 'final_train_acc', 'final_test_loss', 'final_test_acc'])

if metrics_available:
    # Filter out runs without models and sort by T, gamma
    valid_df = df[df['model_exists'] == True].copy()
    valid_df = valid_df.sort_values(['T', 'gamma']).reset_index(drop=True)
    
    if len(valid_df) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(18, 12))
        fig.suptitle('CIFAR-10 Classification Results Comparison', fontsize=16)
        
        # Use parameter labels for x-axis
        x_labels = valid_df['param_label']
        
        # Plot training loss
        if 'final_train_loss' in valid_df.columns:
            axes[0,0].bar(range(len(valid_df)), valid_df['final_train_loss'], color='skyblue')
            axes[0,0].set_title('Final Training Loss')
            axes[0,0].set_xlabel('Parameters (T, γ)')
            axes[0,0].set_ylabel('Loss')
            axes[0,0].set_xticks(range(len(valid_df)))
            axes[0,0].set_xticklabels(x_labels, rotation=45, ha='right')
        
        # Plot training accuracy
        if 'final_train_acc' in valid_df.columns:
            axes[0,1].bar(range(len(valid_df)), valid_df['final_train_acc'], color='lightgreen')
            axes[0,1].set_title('Final Training Accuracy')
            axes[0,1].set_xlabel('Parameters (T, γ)')
            axes[0,1].set_ylabel('Accuracy (%)')
            axes[0,1].set_xticks(range(len(valid_df)))
            axes[0,1].set_xticklabels(x_labels, rotation=45, ha='right')
        
        # Plot test loss
        if 'final_test_loss' in valid_df.columns:
            axes[1,0].bar(range(len(valid_df)), valid_df['final_test_loss'], color='salmon')
            axes[1,0].set_title('Final Test Loss')
            axes[1,0].set_xlabel('Parameters (T, γ)')
            axes[1,0].set_ylabel('Loss')
            axes[1,0].set_xticks(range(len(valid_df)))
            axes[1,0].set_xticklabels(x_labels, rotation=45, ha='right')
        
        # Plot test accuracy
        if 'final_test_acc' in valid_df.columns:
            axes[1,1].bar(range(len(valid_df)), valid_df['final_test_acc'], color='gold')
            axes[1,1].set_title('Final Test Accuracy')
            axes[1,1].set_xlabel('Parameters (T, γ)')
            axes[1,1].set_ylabel('Accuracy (%)')
            axes[1,1].set_xticks(range(len(valid_df)))
            axes[1,1].set_xticklabels(x_labels, rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()
        
        # Statistical summary
        print("\nStatistical Summary:")
        print("=" * 40)
        numeric_cols = valid_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            print(valid_df[numeric_cols].describe())
else:
    print("No metrics found in checkpoint files. Only parameters and model existence status available.")

In [ ]:
# Create a formatted results table
print("\n" + "=" * 80)
print("CIFAR-10 CLASSIFICATION RESULTS SUMMARY")
print("=" * 80)

# Show parameter variations if available
param_cols = ['T', 'gamma', 'lr', 'weight_decay', 'epochs', 'batch_size']
available_params = [col for col in param_cols if col in df.columns]

if available_params:
    print("\nParameter Variations:")
    print("-" * 40)
    for param in available_params:
        unique_vals = df[param].dropna().unique()
        print(f"{param}: {unique_vals}")

# Final summary table with parameter labels
print("\nFinal Results Table:")
print("-" * 40)
display_cols = ['param_label', 'T', 'gamma', 'model_exists']
if metrics_available:
    metric_cols = ['final_train_loss', 'final_train_acc', 'final_test_loss', 'final_test_acc']
    display_cols.extend([col for col in metric_cols if col in df.columns])

final_table = df[display_cols].sort_values(['T', 'gamma'])
print(final_table.to_string(index=False))

# Create a focused summary table grouped by parameters
print("\n" + "=" * 80)
print("PERFORMANCE BY PARAMETER COMBINATIONS")
print("=" * 80)

if metrics_available:
    # Filter valid results and group by T and gamma
    valid_df = df[df['model_exists'] == True].copy()
    if len(valid_df) > 0:
        summary_by_params = valid_df.groupby(['T', 'gamma']).agg({
            'final_train_loss': ['mean', 'std', 'count'],
            'final_train_acc': ['mean', 'std', 'count'],
            'final_test_loss': ['mean', 'std', 'count'],
            'final_test_acc': ['mean', 'std', 'count']
        }).round(3)
        
        print("\nAggregated Results by (T, γ):")
        print(summary_by_params)

In [ ]:
# Save results to CSV for further analysis
output_file = "cifar10_classification_results_comparison.csv"
df.to_csv(output_file, index=False)
print(f"\nResults saved to {output_file}")

# Save summary statistics if available
if metrics_available:
    valid_df = df[df['model_exists'] == True]
    if len(valid_df) > 0:
        summary_stats = valid_df.describe()
        summary_stats.to_csv("cifar10_classification_summary_stats.csv")
        print("Summary statistics saved to cifar10_classification_summary_stats.csv")

In [ ]:
# Add a heatmap visualization for parameter performance
if metrics_available:
    valid_df = df[df['model_exists'] == True].copy()
    if len(valid_df) > 0:
        # Create pivot tables for heatmap visualization
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Parameter Performance Heatmaps', fontsize=16)
        
        # Training Loss Heatmap
        if 'final_train_loss' in valid_df.columns:
            train_loss_pivot = valid_df.pivot_table(values='final_train_loss', index='T', columns='gamma', aggfunc='mean')
            sns.heatmap(train_loss_pivot, annot=True, fmt='.3f', cmap='RdYlBu_r', ax=axes[0,0])
            axes[0,0].set_title('Training Loss by T and γ')
            axes[0,0].set_xlabel('γ (gamma)')
            axes[0,0].set_ylabel('T')
        
        # Training Accuracy Heatmap
        if 'final_train_acc' in valid_df.columns:
            train_acc_pivot = valid_df.pivot_table(values='final_train_acc', index='T', columns='gamma', aggfunc='mean')
            sns.heatmap(train_acc_pivot, annot=True, fmt='.1f', cmap='viridis', ax=axes[0,1])
            axes[0,1].set_title('Training Accuracy by T and γ')
            axes[0,1].set_xlabel('γ (gamma)')
            axes[0,1].set_ylabel('T')
        
        # Test Loss Heatmap
        if 'final_test_loss' in valid_df.columns:
            test_loss_pivot = valid_df.pivot_table(values='final_test_loss', index='T', columns='gamma', aggfunc='mean')
            sns.heatmap(test_loss_pivot, annot=True, fmt='.3f', cmap='RdYlBu_r', ax=axes[1,0])
            axes[1,0].set_title('Test Loss by T and γ')
            axes[1,0].set_xlabel('γ (gamma)')
            axes[1,0].set_ylabel('T')
        
        # Test Accuracy Heatmap
        if 'final_test_acc' in valid_df.columns:
            test_acc_pivot = valid_df.pivot_table(values='final_test_acc', index='T', columns='gamma', aggfunc='mean')
            sns.heatmap(test_acc_pivot, annot=True, fmt='.1f',cmap='viridis', ax=axes[1,1])
            axes[1,1].set_title('Test Accuracy by T and γ')
            axes[1,1].set_xlabel('γ (gamma)')
            axes[1,1].set_ylabel('T')
        
        plt.tight_layout()
        plt.show()
        
        # Find best parameter combinations
        print("\nBest Parameter Combinations:")
        print("=" * 50)
        
        best_train_acc = valid_df.loc[valid_df['final_train_acc'].idxmax()]
        best_test_acc = valid_df.loc[valid_df['final_test_acc'].idxmax()]
        best_train_loss = valid_df.loc[valid_df['final_train_loss'].idxmin()]
        best_test_loss = valid_df.loc[valid_df['final_test_loss'].idxmin()]
        
        print(f"Best Training Accuracy: {best_train_acc['final_train_acc']:.2f}% at T={best_train_acc['T']:.0f}, γ={best_train_acc['gamma']:.2f}")
        print(f"Best Test Accuracy: {best_test_acc['final_test_acc']:.2f}% at T={best_test_acc['T']:.0f}, γ={best_test_acc['gamma']:.2f}")
        print(f"Best Training Loss: {best_train_loss['final_train_loss']:.3f} at T={best_train_loss['T']:.0f}, γ={best_train_loss['gamma']:.2f}")
        print(f"Best Test Loss: {best_test_loss['final_test_loss']:.3f} at T={best_test_loss['T']:.0f}, γ={best_test_loss['gamma']:.2f}")

In [ ]:
# Create heatmap-style tables for each metric
if metrics_available:
    valid_df = df[df['model_exists'] == True].copy()
    if len(valid_df) > 0:
        print("="*80)
        print("HEATMAP TABLES: PERFORMANCE BY T AND γ PARAMETERS")
        print("="*80)
        
        # Training Loss Table
        if 'final_train_loss' in valid_df.columns:
            print("\n📊 TRAINING LOSS")
            print("-" * 40)
            train_loss_pivot = valid_df.pivot_table(values='final_train_loss', index='T', columns='gamma', aggfunc='mean')
            print(train_loss_pivot.round(4))
            
        # Training Accuracy Table
        if 'final_train_acc' in valid_df.columns:
            print("\n📊 TRAINING ACCURACY (%)")
            print("-" * 40)
            train_acc_pivot = valid_df.pivot_table(values='final_train_acc', index='T', columns='gamma', aggfunc='mean')
            print(train_acc_pivot.round(2))
            
        # Test Loss Table
        if 'final_test_loss' in valid_df.columns:
            print("\n📊 TEST LOSS")
            print("-" * 40)
            test_loss_pivot = valid_df.pivot_table(values='final_test_loss', index='T', columns='gamma', aggfunc='mean')
            print(test_loss_pivot.round(4))
            
        # Test Accuracy Table
        if 'final_test_acc' in valid_df.columns:
            print("\n📊 TEST ACCURACY (%)")
            print("-" * 40)
            test_acc_pivot = valid_df.pivot_table(values='final_test_acc', index='T', columns='gamma', aggfunc='mean')
            print(test_acc_pivot.round(2))
            
        # Summary of best values in each table
        print("\n" + "="*80)
        print("BEST VALUES SUMMARY")
        print("="*80)
        
        if 'final_train_loss' in valid_df.columns:
            min_train_loss = train_loss_pivot.min().min()
            best_train_loss_pos = train_loss_pivot.stack().idxmin()
            print(f"🏆 Best Training Loss: {min_train_loss:.4f} at T={best_train_loss_pos[0]:.0f}, γ={best_train_loss_pos[1]:.2f}")
        
        if 'final_train_acc' in valid_df.columns:
            max_train_acc = train_acc_pivot.max().max()
            best_train_acc_pos = train_acc_pivot.stack().idxmax()
            print(f"🏆 Best Training Accuracy: {max_train_acc:.2f}% at T={best_train_acc_pos[0]:.0f}, γ={best_train_acc_pos[1]:.2f}")
        
        if 'final_test_loss' in valid_df.columns:
            min_test_loss = test_loss_pivot.min().min()
            best_test_loss_pos = test_loss_pivot.stack().idxmin()
            print(f"🏆 Best Test Loss: {min_test_loss:.4f} at T={best_test_loss_pos[0]:.0f}, γ={best_test_loss_pos[1]:.2f}")
        
        if 'final_test_acc' in valid_df.columns:
            max_test_acc = test_acc_pivot.max().max()
            best_test_acc_pos = test_acc_pivot.stack().idxmax()
            print(f"🏆 Best Test Accuracy: {max_test_acc:.2f}% at T={best_test_acc_pos[0]:.0f}, γ={best_test_acc_pos[1]:.2f}")
        
        # Parameter performance analysis
        print("\n" + "="*80)
        print("PARAMETER PERFORMANCE ANALYSIS")
        print("="*80)
        
        if 'final_test_acc' in valid_df.columns:
            print("\n📈 Test Accuracy by T (averaged over γ):")
            print("-" * 40)
            acc_by_T = valid_df.groupby('T')['final_test_acc'].mean().round(2)
            print(acc_by_T)
            
            print("\n📈 Test Accuracy by γ (averaged over T):")
            print("-" * 40)
            acc_by_gamma = valid_df.groupby('gamma')['final_test_acc'].mean().round(2)
            print(acc_by_gamma)
        
        # Create a formatted summary table with styled output
        print("\n" + "="*80)
        print("FORMATTED SUMMARY TABLE")
        print("="*80)
        
        # Create a comprehensive summary
        summary_data = []
        for _, row in valid_df.iterrows():
            summary_data.append({
                'T': int(row['T']),
                'γ': row['gamma'],
                'Train Loss': f"{row['final_train_loss']:.4f}",
                'Train Acc': f"{row['final_train_acc']:.2f}%",
                'Test Loss': f"{row['final_test_loss']:.4f}",
                'Test Acc': f"{row['final_test_acc']:.2f}%"
            })
        
        summary_table = pd.DataFrame(summary_data)
        summary_table = summary_table.sort_values(['T', 'γ'])
        print(summary_table.to_string(index=False))